# Train skin classifier v3 (ISIC 2019) on Google Colab

**Goal:** Fine-tune EfficientNet-B0 from your **v2** weights → save `skin_classifier_v3.pth` → download to your Mac.

## Before you start
1. In Colab menu: **Runtime → Change runtime type → Hardware accelerator: GPU** (T4 is fine).
2. On your Mac, copy `models/skin_classifier_v2.pth` to Google Drive (e.g. `My Drive/derma-rag/skin_classifier_v2.pth`).  
   (v2 is **not** on GitHub — too large / gitignored.)
3. You can **stop** the ISIC download on your Mac if you use Colab to download instead (faster).

**Time on free T4:** download ~10–20 min, prepare ~5 min, train ~30–60 min (12 epochs).

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Clone project from GitHub

In [ ]:
REPO = "https://github.com/saleemaltout24/derma-rag.git"
!rm -rf /content/derma-rag
!git clone --depth 1 {REPO} /content/derma-rag
%cd /content/derma-rag

In [ ]:
# Ensure ISIC 2019 CSV uses column "AK" (not only AKIEC)
from pathlib import Path
prep = Path("scripts/prepare_isic2019.py")
t = prep.read_text()
if '"AK": "AK"' not in t:
    t = t.replace('"AKIEC": "AK",', '"AK": "AK",\n    "AKIEC": "AK",')
    prep.write_text(t)
    print("Patched prepare_isic2019.py for AK column")
else:
    print("prepare_isic2019.py OK")

## 2. Install packages (minimal — training only)

In [ ]:
!pip install -q pillow torchvision

## 3. Download ISIC 2019 training data (~9.1 GB)

Runs on Colab servers (usually faster than home Wi‑Fi). Skip this cell if you already have the zip in Drive (see optional cell below).

In [ ]:
import os
from pathlib import Path

RAW = Path("data/isic2019/raw")
RAW.mkdir(parents=True, exist_ok=True)

BASE = "https://isic-archive.s3.amazonaws.com/challenges/2019"
files = [
    "ISIC_2019_Training_GroundTruth.csv",
    "ISIC_2019_Training_Metadata.csv",
    "ISIC_2019_Training_Input.zip",
]
for name in files:
    dest = RAW / name
    if dest.name.endswith(".zip") and dest.exists() and dest.stat().st_size > 9_000_000_000:
        print(f"Skip (already have): {name}")
        continue
    if dest.exists() and not dest.name.endswith(".zip"):
        print(f"Skip: {name}")
        continue
    print(f"Downloading {name} ...")
    !curl -L -o "{dest}" "{BASE}/{name}"

zip_path = RAW / "ISIC_2019_Training_Input.zip"
print("Zip size GB:", round(zip_path.stat().st_size / 1e9, 2))

In [ ]:
# Unzip images (one-time, ~5–10 min)
import zipfile
from pathlib import Path

RAW = Path("data/isic2019/raw")
zip_path = RAW / "ISIC_2019_Training_Input.zip"
out_dir = RAW / "ISIC_2019_Training_Input"
if not out_dir.is_dir():
    print("Unzipping ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(RAW)
    print("Done.")
else:
    n = len(list(out_dir.glob("*.jpg")))
    print(f"Already unzipped ({n} jpgs in folder).")

### Optional: use ISIC zip from Google Drive instead

If you already downloaded the zip to Drive, set `DRIVE_ZIP` and run this cell **instead of** the download cell above.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# DRIVE_ZIP = "/content/drive/MyDrive/ISIC_2019_Training_Input.zip"  # <-- your path
# !mkdir -p data/isic2019/raw
# !cp "{DRIVE_ZIP}" data/isic2019/raw/
# Then run the unzip cell above.

## 4. Copy v2 weights from Google Drive

Upload `skin_classifier_v2.pth` from your Mac to Drive first.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

# CHANGE THIS to where you put v2 on Drive:
DRIVE_V2 = "/content/drive/MyDrive/derma-rag/skin_classifier_v2.pth"

dest = Path("models/skin_classifier_v2.pth")
dest.parent.mkdir(parents=True, exist_ok=True)

src = Path(DRIVE_V2)
if not src.is_file():
    raise FileNotFoundError(
        f"Not found: {DRIVE_V2}\n"
        "Upload skin_classifier_v2.pth from your Mac to Google Drive and fix DRIVE_V2."
    )
shutil.copy2(src, dest)
print(f"Copied v2 -> {dest} ({dest.stat().st_size / 1e6:.1f} MB)")

### Alternative: upload v2 directly in Colab (no Drive)

Run the cell below, click **Choose Files**, pick `skin_classifier_v2.pth` from your Mac (~16 MB).

In [ ]:
# from google.colab import files
# from pathlib import Path
# uploaded = files.upload()  # pick skin_classifier_v2.pth
# Path("models").mkdir(exist_ok=True)
# for name, data in uploaded.items():
#     Path("models/skin_classifier_v2.pth").write_bytes(data)
# print("Saved models/skin_classifier_v2.pth")

## 5. Build train/val folders

In [ ]:
!python scripts/prepare_isic2019.py

## 6. Train v3 (GPU)

Watch **AK val acc** and **SCC val acc** each epoch.

In [ ]:
!python scripts/train_skin_classifier.py \
  --epochs 12 \
  --batch-size 64 \
  --num-workers 2 \
  --lr 1e-4 \
  --pretrained models/skin_classifier_v2.pth \
  --output models/skin_classifier_v3.pth

## 7. Download v3 to your Mac

In [ ]:
from pathlib import Path
from google.colab import files

v3 = Path("models/skin_classifier_v3.pth")
if not v3.is_file():
    raise FileNotFoundError("Training did not create v3. Check errors above.")
print(f"Downloading {v3} ({v3.stat().st_size / 1e6:.1f} MB)")
files.download(str(v3))

## 8. Use v3 on your Mac

1. Move the downloaded file to:
   ```
   /Users/mac/Desktop/htmlcss/derma-rag/models/skin_classifier_v3.pth
   ```
2. Restart backend:
   ```bash
   cd /Users/mac/Desktop/htmlcss/derma-rag
   source venv/bin/activate
   uvicorn app:app --host 127.0.0.1 --port 8000 --reload
   ```
3. Test your AK photos in the browser and run:
   ```bash
   python scripts/eval_classifier.py
   ```

The app uses **v3 automatically** when `models/skin_classifier_v3.pth` exists.